Downloading image and libreries

In [8]:
!pip install pillow

from PIL import Image
from google.colab import files
import math



XOR Encryption

 Note: XOR with a repeating key is a simple cipher used here for demonstration. In production, AES would be used

In [9]:
def xor_encrypt(message, key):
    return ''.join(chr(ord(message[i]) ^ ord(key[i % len(key)])) for i in range(len(message)))

Binary conversion

In [10]:
def text_to_binary(text):
    return ''.join(format(ord(c), '08b') for c in text)

def binary_to_text(binary):
    chars = [binary[i:i+8] for i in range(0, len(binary), 8)]
    return ''.join(chr(int(c, 2)) for c in chars)



PSNR: measures how different the stego image is from the original. Above 40 dB = invisible to the human eye

In [11]:
def calculate_psnr(original_path, stego_path):
    original = Image.open(original_path).convert("RGB")
    stego = Image.open(stego_path).convert("RGB")

    orig_pixels = list(original.getdata())
    stego_pixels = list(stego.getdata())

    mse = sum(
        (o[c] - s[c]) ** 2
        for o, s in zip(orig_pixels, stego_pixels)
        for c in range(3)
    ) / (len(orig_pixels) * 3)

    if mse == 0:
        print("📊 PSNR: ∞ dB (images are perfectly identical)")
        return

    psnr = 10 * math.log10(255**2 / mse)
    print(f"📊 PSNR: {psnr:.2f} dB  (above 40 dB = imperceptible to the human eye)")



Encode : hidding the message inside the image

In [12]:
def encode_image(image_path, message, key, output_path):
    image = Image.open(image_path).convert("RGB")
    pixels = list(image.getdata())

    # Encrypt the message then add the stop marker
    encrypted = xor_encrypt(message + "###END###", key)
    binary_message = text_to_binary(encrypted)

    # Check the message fits inside the image
    max_bits = len(pixels) * 3
    required_bits = len(binary_message)
    print(f"📦 Image capacity : {max_bits // 8} characters")
    print(f"📨 Message needs  : {required_bits // 8} characters")

    if required_bits > max_bits:
        raise ValueError("❌ Message is too long for this image. Use a larger image or a shorter message.")

    # Embed each bit of the message into the LSB of each color channel
    new_pixels = []
    msg_index = 0

    for pixel in pixels:
        r, g, b = pixel

        if msg_index < len(binary_message):
            r = (r & ~1) | int(binary_message[msg_index])
            msg_index += 1

        if msg_index < len(binary_message):
            g = (g & ~1) | int(binary_message[msg_index])
            msg_index += 1

        if msg_index < len(binary_message):
            b = (b & ~1) | int(binary_message[msg_index])
            msg_index += 1

        new_pixels.append((r, g, b))

    image.putdata(new_pixels)

    # Save as PNG — JPEG compression would destroy the hidden bits
    print("💾 Saving as PNG (lossless) — JPEG compression would destroy the hidden bits.")
    image.save(output_path)
    print("✅ Message encoded successfully!")

Decode : Extract the hidden message from the image

In [13]:
def decode_image(image_path, key):
    image = Image.open(image_path).convert("RGB")
    pixels = list(image.getdata())

    # Read the LSB of every color channel back into a binary string
    binary_data = ""
    for pixel in pixels:
        r, g, b = pixel
        binary_data += str(r & 1)
        binary_data += str(g & 1)
        binary_data += str(b & 1)

    # Convert binary back to characters, stop when we hit ###END###
    all_bytes = [binary_data[i:i+8] for i in range(0, len(binary_data), 8)]

    raw_message = ""
    for byte in all_bytes:
        char = chr(int(byte, 2))
        raw_message += char
        if "###END###" in raw_message:
            break

    # FIX: cut everything at the stop marker before decrypting
    if "###END###" not in raw_message:
        print("❌ No hidden message found, or wrong password.")
        return

    clean_message = raw_message[:raw_message.index("###END###")]
    decrypted = xor_encrypt(clean_message, key)
    print("🔓 Hidden message:", decrypted)


Main flow

In [14]:
print("📤 Upload your image:")
uploaded = files.upload()
image_name = list(uploaded.keys())[0]

message = input("Enter your secret message: ")
key     = input("Enter password: ")

output_image = "output.png"

# Step 1: Encode
encode_image(image_name, message, key, output_image)

# Step 2: Download the stego image
files.download(output_image)

# Step 3: Measure image quality (PSNR)
print("\n📐 Measuring image quality...")
calculate_psnr(image_name, output_image)

# Step 4: Decode to verify the message came back correctly
print("\n🔍 Decoding to verify...")
decode_image(output_image, key)

📤 Upload your image:


Saving white.jpg to white.jpg
Enter your secret message: lets do a quick test with some long sentences a fake message no one cares about talking about nothing serious END testing the white photo will it show that there is an encrypted message in it or not for some reason im hoping this fails and i start seiing some colors in the white bc somehow my mind is telling me that it will show in the white now lets END hope for the best and see what output will i get
Enter password: mila jaw
📦 Image capacity : 270000 characters
📨 Message needs  : 414 characters
💾 Saving as PNG (lossless) — JPEG compression would destroy the hidden bits.
✅ Message encoded successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📐 Measuring image quality...
📊 PSNR: 78.02 dB  (above 40 dB = imperceptible to the human eye)

🔍 Decoding to verify...
❌ No hidden message found, or wrong password.
